# SAPP representative days rebuilt from the 2024 hourly load

This notebook rebuilds the EPM time structure of `data_sapp_nuc_rsa_moz`: `pHours`, `pDemandProfile` and `pVREProfile`, plus the two other files indexed by representative day, `pVREgenProfile` and `pTradePrice`. It uses the 2024 hourly load sent by SAPP (`Load_data_SAPP.xlsx`) and ERA5 solar and wind series for the same year, run through `representativedays_pipeline.py`.

**Why.** The input check flagged two problems in the current demand profiles:

- **Profile maximum below 1** in 16 zones (0.81 in Zimbabwe). EPM builds hourly demand as profile × forecast peak, so the model peak was up to 19% below the forecast peak, and the planning reserve was sized on that lower peak.
- **Negative hourly demand** in DRC and the three Mozambique zones. The forecast's load factor is much lower than the profile's, and the energy correction in `generate_demand.gms` pushes the low hours below zero.

**Correction applied in the pipeline.** The pipeline already adds a `max_Load` special day, but it is the day closest to the centre of the highest-load cluster, computed on load **summed over all zones**. It follows South Africa and rarely contains anyone's peak hour. The new option `peak_load_days=True` adds the actual peak day of each country as a special day of weight 1.

The notebook writes candidate EPM files to `output/sapp_2024/epm/`. It does not change the model's input folder.

## 0. Settings

In [ ]:
import os, re, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path.cwd()
if not (BASE / 'representativedays_pipeline.py').exists():
    BASE = BASE / 'pre-analysis' / 'representative_days'
os.chdir(BASE)                      # the pipeline resolves gams/ from the working directory
sys.path.insert(0, str(BASE.parent))
from representative_days.representativedays_pipeline import run_representative_days_pipeline

ROOT = BASE.parents[1]
EPM_INPUT = ROOT / 'epm' / 'input' / 'data_sapp_nuc_rsa_moz'      # read only
LOAD_XLSX = BASE / 'input' / 'sapp' / 'Load_data_SAPP.xlsx'       # copy of the SAPP file
ERA5_2024 = ROOT / 'pre-analysis' / 'climate_years' / 'output' / 'sapp_2024' / 'hourly'
ERA5_HIST = ROOT / 'pre-analysis' / 'climate_years' / 'output' / 'sapp' / 'hourly'
OUT = BASE / 'output' / 'sapp_2024'
EPM_OUT = OUT / 'epm'
EPM_OUT.mkdir(parents=True, exist_ok=True)

YEAR = 2024
SEASONS = {m: (m - 1) // 3 + 1 for m in range(1, 13)}   # Q1 = Jan-Mar ... Q4 = Oct-Dec, as the current pHours
N_REPR_DAYS = 2                 # representative days per season chosen by the optimisation (special days come on top)
BLACKOUT_THRESHOLD = 0.5        # an hour below 50% of the median of the same hour over +-7 days is treated as an outage

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3})
warnings.filterwarnings('ignore', category=FutureWarning)

## 1. Read the SAPP hourly load

The workbook has one block of columns per country. Where a block has both **Original** and **Adapted** columns, *Adapted* is SAPP's corrected series and is used here. South Africa has only *Original*, and Malawi a single *Total* column.

In [ ]:
raw = pd.read_excel(LOAD_XLSX, header=None)
titles = {c: v for c, v in raw.iloc[1].items() if isinstance(v, str)}
header = raw.iloc[2]
blocks = sorted(titles)
load_raw, info = {}, []
for i, c0 in enumerate(blocks):
    c1 = blocks[i + 1] if i + 1 < len(blocks) else raw.shape[1]
    name = re.match(r'([A-Z ]+?)\s+(HOURLY|Hourly|20\d\d)', titles[c0]).group(1).strip()
    country = {'SOUTH AFRICA': 'South_Africa'}.get(name, name.title())
    cols = {header[c]: c for c in range(c0, c1) if isinstance(header[c], str)}
    value_col = next(k for k in ('Adapted', 'Original', 'Total') if k in cols)
    df = pd.DataFrame({'month': raw.iloc[3:, cols['Month']], 'day': raw.iloc[3:, cols['Month-Day']],
                       'hour': raw.iloc[3:, cols['Hour-day']], 'value': raw.iloc[3:, cols[value_col]]})
    if 'Original' in cols and value_col != 'Original':
        df['original'] = raw.iloc[3:, cols['Original']]
    df = df.apply(pd.to_numeric, errors='coerce').dropna(subset=['month', 'day', 'hour', 'value'])
    df = df.astype({'month': int, 'day': int, 'hour': int}).reset_index(drop=True)
    load_raw[country] = df
    info.append(dict(country=country, column_used=value_col, rows=len(df),
                     starts=f"{df.month.iloc[0]:02d}-{df.day.iloc[0]:02d}", hour_labels=f"{df.hour.min()}-{df.hour.max()}",
                     duplicated_hours=int(df.duplicated(['month', 'day', 'hour']).sum()),
                     feb29_hours=int(((df.month == 2) & (df.day == 29)).sum()),
                     hours_corrected_by_SAPP=int((df.original - df.value).abs().gt(1e-6).sum()) if 'original' in df else None,
                     original_min_MW=round(df.original.min(), 1) if 'original' in df else None))
info = pd.DataFrame(info).set_index('country')
info

**What the table shows**

- Hour labels run 1–24, meaning hour ending. They are converted to 0–23 (hour beginning) before the pipeline.
- **Lesotho** starts on 1 April. Its year is April 2024 to March 2025, with 1 April repeated at the end and no 29 February. The duplicate day is dropped and the months are put in calendar order, so January–March come from 2025. Weekdays are therefore shifted for those months, which has little effect on seasonal profiles.
- All other countries are calendar 2024, a leap year: 29 February is dropped to keep 365 days.
- SAPP's *Adapted* columns already remove zeros and negative values from the *Original* data, for example Namibia down to −927 MW and Eswatini and Zambia at 0 MW.

## 2. Clean the series

1. Drop duplicated hours (Lesotho) and 29 February, then sort in calendar order (365 days × 24 hours).
2. Replace outage hours. An hour is treated as a blackout or load-shedding hour when it is below **50%** of the median of the same hour of the day over the surrounding 15 days. It is replaced by that median. A higher threshold starts catching normal low-load hours, such as holidays. Spikes are kept, because they may be real peaks.

In [ ]:
def clean(df):
    df = df.drop_duplicates(['month', 'day', 'hour'], keep='first')
    df = df[~((df.month == 2) & (df.day == 29))].sort_values(['month', 'day', 'hour']).reset_index(drop=True)
    assert len(df) == 8760, len(df)
    M = df.value.to_numpy(dtype=float).reshape(365, 24)
    ref = pd.DataFrame(M).rolling(15, center=True, min_periods=5).median().to_numpy()
    low = M < BLACKOUT_THRESHOLD * ref
    df['clean'] = np.where(low, ref, M).ravel()
    df['replaced'] = low.ravel()
    return df

load = {c: clean(df) for c, df in load_raw.items()}

def stats(v):
    return dict(peak_MW=v.max(), energy_GWh=v.sum() / 1e3, load_factor=v.sum() / (v.max() * len(v)), min_over_peak=v.min() / v.max())

rows = []
for c, df in load.items():
    s_raw, s_clean = stats(df.value), stats(df['clean'])
    pk = df.loc[df['clean'].idxmax()]
    rows.append(dict(country=c, hours_replaced=int(df.replaced.sum()), days_affected=int(df.replaced.values.reshape(365, 24).any(1).sum()),
                     min_over_peak_before=s_raw['min_over_peak'], min_over_peak_after=s_clean['min_over_peak'],
                     peak_MW=s_clean['peak_MW'], energy_GWh=s_clean['energy_GWh'], load_factor=s_clean['load_factor'],
                     peak_time=f"{int(pk.month):02d}-{int(pk.day):02d} h{int(pk.hour)}"))
load_stats = pd.DataFrame(rows).set_index('country').sort_values('load_factor', ascending=False)
load_stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for ax, c in zip(axes, ['Mozambique', 'Tanzania', 'Malawi']):
    df = load[c]
    day = df.replaced.values.reshape(365, 24).sum(1).argmax()
    sl = slice(max(day - 3, 0) * 24, min(day + 4, 365) * 24)
    x = np.arange(sl.start, sl.stop) / 24
    ax.plot(x, df.value.iloc[sl], color='#c0392b', lw=1, label='as received')
    ax.plot(x, df['clean'].iloc[sl], color='#1f4e79', lw=1.4, label='cleaned')
    ax.set_title(f"{c}: week around the largest outage"); ax.set_xlabel('day of year'); ax.set_ylabel('MW')
axes[0].legend(frameon=False)
plt.tight_layout()

After cleaning, the lowest hour of every country is a plausible night minimum and no longer an outage. The load factors range from 0.59 in Lesotho to 0.76 in South Africa and Angola.

## 3. Map countries to EPM zones

Each zone takes the profile of its country, normalised to its own annual peak (= 1). The pipeline normalises a whole file by its **overall** maximum, so loads in MW would all be scaled by South Africa's 36 GW. Normalising per country first avoids that.

| Zones | Profile used | Note |
|---|---|---|
| Mozambique_South, _Central, _North | Mozambique | Same national profile for all regions, as in the SAPP data |
| Zimbabwe_East, _West | Zimbabwe | |
| Angola_North, _Central, _South, _East | Angola | |
| DRC | Angola (proxy) | No DRC data. Kinshasa and Luanda have similar climate and urban loads. |
| DRC_South | Zambia (proxy) | No DRC data. Katanga is mining-dominated, like the Copperbelt. |
| Mozal | Flat = 1 | Aluminium smelter running at constant load |

Only the 11 countries with real data enter the day selection. The proxies and Mozal are added when the EPM files are written, so they don't count twice in the clustering.

In [ ]:
ZONE_LOAD_SOURCE = {
    'South_Africa': 'South_Africa', 'Lesotho': 'Lesotho', 'Eswatini': 'Eswatini', 'Botswana': 'Botswana',
    'Namibia': 'Namibia', 'Zambia': 'Zambia', 'Tanzania': 'Tanzania', 'Malawi': 'Malawi',
    'Mozambique_South': 'Mozambique', 'Mozambique_Central': 'Mozambique', 'Mozambique_North': 'Mozambique',
    'Zimbabwe_East': 'Zimbabwe', 'Zimbabwe_West': 'Zimbabwe',
    'Angola_North': 'Angola', 'Angola_Central': 'Angola', 'Angola_South': 'Angola', 'Angola_East': 'Angola',
    'DRC': 'Angola', 'DRC_South': 'Zambia', 'Mozal': 'flat',
}
current_zones = set(pd.read_csv(EPM_INPUT / 'load' / 'pDemandProfile.csv').iloc[:, 0])
assert current_zones == set(ZONE_LOAD_SOURCE), current_zones ^ set(ZONE_LOAD_SOURCE)

calendar = load['South_Africa'][['month', 'day', 'hour']].copy()
calendar['hour'] -= 1             # hour ending 1-24 -> hour beginning 0-23, as the pipeline expects
load_norm = pd.DataFrame({c: df['clean'] / df['clean'].max() for c, df in load.items()})
load_long = pd.concat([calendar.assign(zone=c, value=load_norm[c].round(5)) for c in load_norm], ignore_index=True)
(OUT / 'inputs').mkdir(parents=True, exist_ok=True)
load_long[['zone', 'month', 'day', 'hour', 'value']].to_csv(OUT / 'inputs' / 'load_data.csv', index=False)
print(f"{load_norm.shape[1]} load series written, each with max = {load_norm.max().min():.2f}")

## 4. Solar and wind for the same year

The solar PV and wind capacity factors come from the ERA5 point series of `pre-analysis/climate_years/fetch_climate_years.py`, one site per zone. Using the same year as the load keeps the day-to-day correlation between demand, sun and wind, which representative days are meant to capture.

Two alignments are needed:

- **Time zone.** ERA5 is in UTC, the load is in local time. Angola and western DRC are UTC+1, Tanzania UTC+3, and the rest of SAPP UTC+2.
- **Hour convention.** ERA5 radiation at time *t* is accumulated over the hour ending at *t*. It is shifted one more hour, so that hour 0 means 00:00–01:00 as for the load.

In [ ]:
UTC_OFFSET = {z: 2 for z in ['South_Africa', 'Lesotho', 'Eswatini', 'Mozambique', 'Mozambique_South', 'Mozambique_Central',
                             'Mozambique_North', 'Mozal', 'Zimbabwe_East', 'Zimbabwe_West', 'Botswana', 'Namibia',
                             'Zambia', 'Malawi', 'DRC_South']}
UTC_OFFSET.update({z: 1 for z in ['Angola', 'Angola_North', 'Angola_Central', 'Angola_South', 'Angola_East', 'DRC']})
UTC_OFFSET['Tanzania'] = 3

def era5_local(zone):
    # Hourly PV and wind of one zone site, in local time, 365 days x 24 hours, hour beginning 0-23.
    src, year = (ERA5_2024, YEAR) if (ERA5_2024 / f'{zone}.csv.gz').exists() else (ERA5_HIST, 2023)
    d = pd.read_csv(src / f'{zone}.csv.gz', parse_dates=['time'])
    d = d[d.time.dt.year == year].sort_values('time').reset_index(drop=True)
    off = UTC_OFFSET[zone]
    t = pd.date_range(f'{year}-01-01', periods=len(d), freq='h')
    out = pd.DataFrame({'month': t.month, 'day': t.day, 'hour': t.hour,
                        'PV': np.roll(d.PV.to_numpy(), off - 1),      # accumulated over the previous hour
                        'Wind': np.roll(d.Wind.to_numpy(), off)})
    out = out[~((out.month == 2) & (out.day == 29))].reset_index(drop=True)
    return out, year

vre_sites = sorted(UTC_OFFSET)
vre, vre_year = {}, {}
for z in vre_sites:
    vre[z], vre_year[z] = era5_local(z)
years_used = pd.Series(vre_year).value_counts()
print('ERA5 year used:', dict(years_used))
if (years_used.index != YEAR).any():
    print(f'WARNING: {years_used.drop(YEAR, errors="ignore").sum()} site(s) use 2023 because the 2024 download is not complete.')

To keep the optimisation tractable, the day selection uses one PV and one wind site per country, plus the two DRC zones. That gives 11 load + 15 PV + 15 wind series, and the pipeline adds their pairwise correlations. Profiles for every zone in the current `pVREProfile` are written afterwards from the selected days.

In [ ]:
SELECT_SITES = ['South_Africa', 'Lesotho', 'Eswatini', 'Botswana', 'Namibia', 'Zambia', 'Tanzania', 'Malawi',
                'Mozambique_South', 'Mozambique_North', 'Zimbabwe_West', 'Angola_North', 'Angola_South', 'DRC', 'DRC_South']
for tech in ('PV', 'Wind'):
    long = pd.concat([vre[z][['month', 'day', 'hour']].assign(zone=z, value=vre[z][tech].round(5)) for z in SELECT_SITES],
                     ignore_index=True)
    long[['zone', 'month', 'day', 'hour', 'value']].to_csv(OUT / 'inputs' / f'{tech.lower()}_data.csv', index=False)
input_files = {'PV': str(OUT / 'inputs' / 'pv_data.csv'), 'Wind': str(OUT / 'inputs' / 'wind_data.csv'),
               'Load': str(OUT / 'inputs' / 'load_data.csv')}
cf = pd.DataFrame({tech: {z: vre[z][tech].mean() for z in SELECT_SITES} for tech in ('PV', 'Wind')})
cf.T

## 5. Run the representative-days pipeline

The pipeline runs twice on the same inputs:

- **default**: the pipeline as before, with special days for minimum PV, minimum wind and maximum load (system-wide cluster) in each season, plus 2 optimised days per season;
- **peak days**: the same, with `peak_load_days=True`, which adds the actual peak day of each country.

The comparison isolates the effect of the correction.

In [ ]:
common = dict(seasons_map=SEASONS, input_files=input_files, n_representative_days=N_REPR_DAYS, year_label=YEAR,
              gams_main_file=str(BASE / 'gams' / 'OptimizationModelZone.gms'), verbose=False)
res = {}
for label, flag in (('default', False), ('peak days', True)):
    res[label] = run_representative_days_pipeline(output_dir=OUT / f"run_{label.replace(' ', '_')}",
                                                  peak_load_days=flag, **common)
    rd = res[label]['repr_days']
    print(f"{label:10s}: {len(rd)} days, weight {rd.weight.sum():.0f}, by season {rd.groupby('season').size().to_dict()}")

In [ ]:
# Calendar date of each (season, day index)
dates = pd.DataFrame({'date': pd.date_range(f'{YEAR}-01-01', f'{YEAR}-12-31')})
dates = dates[~((dates.date.dt.month == 2) & (dates.date.dt.day == 29))]
dates['season'] = 'Q' + dates.date.dt.month.map(SEASONS).astype(str)
dates['day'] = dates.groupby('season').cumcount() + 1
date_of = dates.set_index(['season', 'day']).date

def describe_days(rd):
    rd = rd.copy()
    rd['date'] = [date_of[(s, d)].strftime('%d %b') for s, d in zip(rd.season, rd.day)]
    return rd[['season', 'daytype', 'date', 'weight', 'rule']].sort_values(['season', 'daytype']).reset_index(drop=True)

repr_peak = describe_days(res['peak days']['repr_days'])
repr_peak

In [ ]:
pk = res['peak days']['peak_load_days'].copy()
pk['country'] = pk.series.str.replace('Load_', '', regex=False)
pk['date'] = [date_of[(f'Q{s}', d)].strftime('%d %b') for s, d in zip(pk.season, pk.day)]
pk[['country', 'date', 'hour', 'action']]

Each country's peak day is now one of the selected days. Countries that peak on the same day share it, and a peak day that was already selected is not added twice. When the peak day belongs to a cluster already represented by a special day, one day of weight moves from that special day to the peak day, so the season keeps its number of days.

## 6. Check the result

### 6.1 Profile maximum, the main correction

The model peak of each zone equals the profile maximum × the forecast peak. Values below 1 under-state the peak.

In [ ]:
def hourly_by_day(series, rd):
    # series indexed like `calendar` (365 x 24) -> array (n_days, 24) for the selected days
    M = np.asarray(series, dtype=float).reshape(365, 24)
    idx = [dates.index.get_loc(dates[(dates.season == s) & (dates.day == d)].index[0]) for s, d in zip(rd.season, rd.day)]
    return M[idx]

cur_prof = pd.read_csv(EPM_INPUT / 'load' / 'pDemandProfile.csv')
cur_prof.columns = ['zone', 'q', 'd'] + [f't{i}' for i in range(1, 25)]
cur_hours = pd.read_csv(EPM_INPUT / 'pHours.csv', encoding='utf-8-sig')
tcols = [f't{i}' for i in range(1, 25)]

def weighted_lf(P, w):
    return float((P * w[:, None]).sum() / (w.sum() * 24) / P.max())

rows = []
for c in load_norm:
    zone = next(z for z, s in ZONE_LOAD_SOURCE.items() if s == c)
    cp = cur_prof[cur_prof.zone == zone].merge(cur_hours, on=['q', 'd'], suffixes=('', '_w'))
    row = dict(country=c, current=cp[tcols].to_numpy().max())
    for label in res:
        rd = res[label]['repr_days']
        row[label] = hourly_by_day(load_norm[c], rd).max()
    w = res['peak days']['repr_days'].weight.to_numpy()
    row.update(LF_observed=load_stats.loc[c, 'load_factor'],
               LF_current=float((cp[tcols].to_numpy() * cp[[f'{t}_w' for t in tcols]].to_numpy()).sum() / (8760 * cp[tcols].to_numpy().max())),
               LF_new=weighted_lf(hourly_by_day(load_norm[c], res['peak days']['repr_days']), w))
    rows.append(row)
peak_check = pd.DataFrame(rows).set_index('country')
peak_check.style.format('{:.3f}').background_gradient(subset=['current', 'default', 'peak days'], cmap='RdYlGn', vmin=0.8, vmax=1.0)

- **Profile maximum.** With the correction, every country's profile reaches 1.00, so EPM keeps the forecast peak of each zone. The current file ranges from 0.81 to 1.03. The default pipeline also stays below 1 in most countries, which confirms that the system-wide `max_Load` special day isn't enough.
- **Load factor.** `LF_new` is the load factor of the representative profile. It is much closer to the observed 2024 load factor than the current profile is: the current profiles were too flat, which under-stated how peaky demand is.

### 6.2 Load duration curves: full year against the representative days

In [ ]:
rd = res['peak days']['repr_days']; w = rd.weight.to_numpy()
fig, axes = plt.subplots(2, 4, figsize=(15, 6), sharey=True)
for ax, c in zip(axes.ravel(), ['South_Africa', 'Zimbabwe', 'Zambia', 'Mozambique', 'Angola', 'Tanzania', 'Eswatini', 'Lesotho']):
    full = np.sort(load_norm[c].to_numpy())[::-1]
    P = hourly_by_day(load_norm[c], rd)
    vals = P.ravel(); wt = np.repeat(w, 24)
    o = np.argsort(vals)[::-1]
    ax.plot(np.arange(1, 8761), full, color='#999999', lw=2, label='2024, 8,760 h')
    ax.step(np.cumsum(wt[o]), vals[o], where='post', color='#1f4e79', lw=1.3, label='representative days')
    ax.set_title(c); ax.set_xlim(0, 8760); ax.set_ylim(0.2, 1.02)
axes[0, 0].legend(frameon=False, fontsize=8); axes[0, 0].set_ylabel('load / annual peak'); axes[1, 0].set_ylabel('load / annual peak')
plt.tight_layout()

### 6.3 Average day by season

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=True)
c = 'South_Africa'
M = load_norm[c].to_numpy().reshape(365, 24)
P = hourly_by_day(load_norm[c], rd)
for ax, q in zip(axes, ['Q1', 'Q2', 'Q3', 'Q4']):
    m = (dates.season == q).to_numpy()
    sel = (rd.season == q).to_numpy()
    ax.plot(range(1, 25), M[m].mean(0), color='#999999', lw=2, label='2024 average')
    ax.plot(range(1, 25), (P[sel] * w[sel, None]).sum(0) / w[sel].sum(), color='#1f4e79', lw=1.4, label='representative days')
    ax.set_title(f'{c}, {q}'); ax.set_xlabel('hour')
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()

## 7. Write the EPM files

The three files must be replaced **together**, because the day labels (`d1`, `d2`...) refer to different calendar days than in the current files.

- **`pHours.csv`**: the weight of each selected day (days per year it stands for).
- **`pDemandProfile.csv`**: the normalised load of each zone on the selected days, with the country-to-zone mapping of step 3.
- **`pVREProfile.csv`**: every (zone, technology) pair of the current file, from the ERA5 series of that zone on the selected days.
  - Each profile is rescaled so that its annual capacity factor equals the current file's, capped at 1. The current file reflects the resource assessment of the study, while ERA5 at a zone centroid gives the day-to-day pattern only.
  - CSP uses the PV series shape. Offshore wind uses the onshore wind series of the zone, which is only an approximation (see the table).
- **`pVREgenProfile.csv`**: Ruacana run-of-river has one value per season, which is repeated on the new days.
- **`pTradePrice.csv`**: the average of the current days of each season, repeated on the new days. The current prices are almost identical across days, and external exchange is off in this dataset.

In [ ]:
rd = res['peak days']['repr_days'].sort_values(['season', 'daytype']).reset_index(drop=True)
w = rd.weight.to_numpy()

# pHours
ph = rd[['season', 'daytype']].rename(columns={'season': 'q', 'daytype': 'd'})
for t in tcols: ph[t] = rd.weight.astype(int)
ph.to_csv(EPM_OUT / 'pHours.csv', index=False, encoding='utf-8-sig')
print('days per season:', ph.groupby('q')['t1'].sum().to_dict(), '| total', int(ph['t1'].sum()))

# pDemandProfile (same header as the current file)
rows = []
for zone, src in ZONE_LOAD_SOURCE.items():
    P = np.ones((len(rd), 24)) if src == 'flat' else hourly_by_day(load_norm[src], rd)
    for (q, d), vals in zip(rd[['season', 'daytype']].values, P):
        rows.append([zone, q, d] + list(np.round(vals, 4)))
pdp = pd.DataFrame(rows, columns=['Zone', 'm', 'd'] + tcols)
pdp.to_csv(EPM_OUT / 'pDemandProfile.csv', index=False)
print('pDemandProfile:', pdp.Zone.nunique(), 'zones, max per zone:', pdp.groupby('Zone')[tcols].max().max(1).round(3).unique())

In [ ]:
cur_vre = pd.read_csv(EPM_INPUT / 'supply' / 'pVREProfile.csv')
cur_vre = cur_vre.merge(cur_hours, on=['q', 'd'], suffixes=('', '_w'))
RESOURCE = {'PV': 'PV', 'CSPPlant': 'PV', 'OnshoreWind': 'Wind', 'OffshoreWind': 'Wind'}
SITE_ALIAS = {'Zimbabwe': 'Zimbabwe_West'}

def scale_to(P, w, target):
    # Factor k so that the weighted mean of min(k*P, 1) equals target (bisection).
    if target <= 0 or P.max() <= 0:
        return 0.0
    lo, hi = 0.0, 50.0
    for _ in range(60):
        k = (lo + hi) / 2
        m = (np.minimum(k * P, 1) * w[:, None]).sum() / (w.sum() * 24)
        lo, hi = (k, hi) if m < target else (lo, k)
    return k

rows, vre_rows = [], []
for (zone, tech), g in cur_vre.groupby(['zone', 'tech']):
    target = float((g[tcols].to_numpy() * g[[f"{t}_w" for t in tcols]].to_numpy()).sum() / 8760)
    site = SITE_ALIAS.get(zone, zone)
    P = hourly_by_day(vre[site][RESOURCE[tech]], rd)
    raw_cf = float((P * w[:, None]).sum() / (w.sum() * 24))
    k = scale_to(P, w, target)
    Ps = np.minimum(k * P, 1)
    for (q, d), vals in zip(rd[['season', 'daytype']].values, Ps):
        vre_rows.append([zone, tech, q, d] + list(np.round(vals, 4)))
    note = []
    if tech == 'OffshoreWind': note.append('onshore wind shape at the zone site')
    if tech == 'CSPPlant': note.append('PV shape')
    if site != zone: note.append(f'site {site}')
    if k > 2.5 or (0 < k < 0.4): note.append('large rescaling: check the site')
    rows.append(dict(zone=zone, tech=tech, target_CF=target, ERA5_CF=raw_cf, scale=k,
                     new_CF=float((Ps * w[:, None]).sum() / (w.sum() * 24)), note='; '.join(note)))
pvre = pd.DataFrame(vre_rows, columns=['zone', 'tech', 'q', 'd'] + tcols)
pvre.to_csv(EPM_OUT / 'pVREProfile.csv', index=False)
vre_check = pd.DataFrame(rows)
vre_check.style.format({'target_CF': '{:.3f}', 'ERA5_CF': '{:.3f}', 'scale': '{:.2f}', 'new_CF': '{:.3f}'}).hide(axis='index')

In [ ]:
# pVREgenProfile: one value per plant and season, repeated on the new days
gp = pd.read_csv(EPM_INPUT / 'supply' / 'pVREgenProfile.csv')
per_season = gp.groupby(['gen', 'q'])[tcols].mean()
gp_new = rd[['season', 'daytype']].rename(columns={'season': 'q', 'daytype': 'd'}).merge(per_season.reset_index(), on='q')
gp_new[['gen', 'q', 'd'] + tcols].to_csv(EPM_OUT / 'pVREgenProfile.csv', index=False)

# pTradePrice: average over the current days of each season, repeated on the new days
tp = pd.read_csv(EPM_INPUT / 'trade' / 'pTradePrice.csv', header=None, skiprows=1)
tp.columns = ['zext', 'q', 'd', 'y'] + [f't{i:02d}' for i in range(1, 25)]
tp_mean = tp.groupby(['zext', 'q', 'y'])[[f't{i:02d}' for i in range(1, 25)]].mean().reset_index()
tp_new = tp_mean.merge(rd[['season', 'daytype']].rename(columns={'season': 'q', 'daytype': 'd'}), on='q')
tp_new = tp_new[['zext', 'q', 'd', 'y'] + [f't{i:02d}' for i in range(1, 25)]].sort_values(['zext', 'y', 'q', 'd'])
with open(EPM_OUT / 'pTradePrice.csv', 'w', newline='') as f:
    f.write(',,,,' + ','.join(f't{i:02d}' for i in range(1, 25)) + '\n')
    tp_new.to_csv(f, header=False, index=False)
print('Written:', sorted(p.name for p in EPM_OUT.glob('*.csv')))

## 8. Effect on the model's hourly demand

The model's hourly demand is recomputed the way `generate_demand.gms` does: profile × forecast peak, plus the correction that matches the annual energy. It is computed for 2025–2055 from the current forecast (`pDemandForecast_sapp_poolplan_v5_mozalout.csv`), first with the current profiles, then with the new ones.

In [ ]:
fc = pd.read_csv(EPM_INPUT / 'load' / 'pDemandForecast_sapp_poolplan_v5_mozalout.csv')
fc = fc.melt(id_vars=['z', 'type'], var_name='y', value_name='v')
fc['y'] = fc.y.astype(int)
fc = fc[(fc.y >= 2025) & (fc.y <= 2055)].pivot_table(index=['z', 'y'], columns='type', values='v').reset_index()

def model_demand(profile, hours, zone, energy_GWh, peak_MW):
    g = profile[profile.zone == zone].merge(hours, on=['q', 'd'], suffixes=('', '_w'))
    p = g[tcols].to_numpy(); wt = g[[f'{t}_w' for t in tcols]].to_numpy()
    pmax = p.max(); S = (p * wt).sum(); A = ((pmax - p) * wt).sum()
    k = (energy_GWh * 1e3 - peak_MW * S) / A if A > 0 else 0.0
    return p * peak_MW + k * (pmax - p)

new_prof = pdp.rename(columns={'Zone': 'zone', 'm': 'q'})
new_hours = ph
out = []
for _, x in fc[fc.Energy > 1].iterrows():
    for label, prof, hrs in (('current', cur_prof, cur_hours), ('new', new_prof, new_hours)):
        h = model_demand(prof, hrs, x.z, x.Energy, x.Peak)
        out.append(dict(zone=x.z, y=x.y, profile=label, peak_ratio=h.max() / x.Peak, min_MW=h.min()))
dem = pd.DataFrame(out)
summary = dem.groupby(['zone', 'profile']).agg(model_peak_vs_forecast=('peak_ratio', 'min'), lowest_hour_MW=('min_MW', 'min')).unstack('profile')
summary.columns = [f'{a} ({b})' for a, b in summary.columns]
summary.style.format('{:.2f}', subset=[c for c in summary.columns if 'peak' in c]).format('{:,.0f}', subset=[c for c in summary.columns if 'lowest' in c])

- **Model peak.** With the new profiles, the model peak equals the forecast peak in every zone (ratio 1.00), where it was 0.81–1.03 before.
- **Negative hours.** These remain in DRC and Mozambique. They come from the **forecast**, not from the profile: its peak is too high for its energy (load factor 0.23–0.37 in Mozambique_North, 0.31–0.34 in DRC). A steeper, more realistic profile makes this worse, because the energy correction has to remove even more.
- **Fix in the forecast.** Mozambique's zones share the national profile, so each zone's peak should be its energy ÷ (national load factor × 8,760). The national load factor from the cleaned 2024 data is computed below, and the same test is run with the corrected peaks.

In [ ]:
lf_moz = load_stats.loc['Mozambique', 'load_factor']
fix = fc[fc.z.str.startswith('Mozambique')].copy()
fix['Peak_consistent'] = fix.Energy * 1e3 / (lf_moz * 8760)
res_fix = []
for _, x in fix.iterrows():
    h = model_demand(new_prof, new_hours, x.z, x.Energy, x.Peak_consistent)
    res_fix.append(dict(zone=x.z, y=x.y, peak_forecast=x.Peak, peak_consistent=x.Peak_consistent, lowest_hour_MW=h.min()))
res_fix = pd.DataFrame(res_fix)
print(f'Mozambique 2024 load factor (cleaned): {lf_moz:.3f}')
res_fix[res_fix.y.isin([2025, 2030, 2040, 2050, 2055])].pivot_table(index='y', columns='zone', values=['peak_forecast', 'peak_consistent', 'lowest_hour_MW']).round(0)

## 9. Summary of corrections and how to use the files

In [ ]:
moz_ok = res_fix.lowest_hour_MW.min() > 0
corrections = pd.DataFrame([
    ('Load data', 'Lesotho year Apr 2024 - Mar 2025 with 1 April twice', 'Duplicate day dropped, months put in calendar order'),
    ('Load data', '29 February (leap year 2024)', 'Dropped in all series (365 days)'),
    ('Load data', f"Outage hours ({int(load_stats.hours_replaced.sum())} h, mainly Mozambique, Tanzania, Malawi)",
     'Replaced by the median of the same hour over +-7 days'),
    ('Load data', 'No hourly data for DRC; Mozal not in the national data', 'DRC <- Angola, DRC_South <- Zambia (proxies); Mozal flat'),
    ('Pipeline', 'Normalisation by the maximum over all zones', 'Each country normalised to its own peak before the pipeline'),
    ('Pipeline', 'max_Load special day = cluster centre of the system-wide load', 'New option peak_load_days=True: peak day of each country added'),
    ('Pipeline', 'ERA5 in UTC, load in local time', 'ERA5 shifted to local time (UTC+1/+2/+3), radiation by one more hour'),
    ('Result', f"Profile max {peak_check['current'].min():.2f}-{peak_check['current'].max():.2f} in the current file",
     f"Profile max {peak_check['peak days'].min():.2f}-{peak_check['peak days'].max():.2f}: model peak = forecast peak"),
    ('Still open', 'Negative hourly demand in DRC and Mozambique (forecast peak too high for its energy)',
     'Mozambique: peak = energy / (%.3f x 8760) removes it%s. DRC: check the forecast source' % (lf_moz, '' if moz_ok else ' (NOT fully)')),
    ('Still open', 'Offshore wind and CSP profiles', 'Approximated from onshore wind and PV shapes, rescaled to current CFs'),
], columns=['area', 'issue', 'correction'])
corrections.style.hide(axis='index')

**To use the files in EPM**

1. Copy `pHours.csv`, `load/pDemandProfile.csv`, `supply/pVREProfile.csv`, `supply/pVREgenProfile.csv` and `trade/pTradePrice.csv` from `output/sapp_2024/epm/` into the input folder **together**, or point `config.csv` at new file names. The day labels changed, so mixing old and new files would pair the wrong days.
2. The number of days per season can differ, because peak days fall in some seasons only. Check `pHours` and run `python epm.py --folder_input data_sapp_nuc_rsa_moz --check_inputs` before a full run.
3. Fix the Mozambique and DRC peaks in the demand forecast (section 8), otherwise the negative hours remain.

**To rerun**, open this notebook from `pre-analysis/representative_days` in the `epm_env` environment. GAMS must be on the PATH.